In [4]:
import json
import os
from pathlib import Path
from sqlalchemy import create_engine, text
import pandas as pd
from tqdm.notebook import tqdm

from premodel2_ver320 import GEOScorer

# 스코어러 객체 생성 (전역 모델 로딩)
scorer = GEOScorer()
print("✅ GEOScorer 모델 로드 완료!")

# 1. 환경변수 파일(.env) 로드
CURRENT_DIR = Path.cwd()
ENV_PATH = CURRENT_DIR / "../project_db/geo.env"

if ENV_PATH.exists():
    with open(ENV_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" in line:
                key, val = line.split("=", 1)
                os.environ[key.strip()] = val.strip()

# 2. DB 연결 정보 추출 및 engine 생성
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

ENGINE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(ENGINE_URL)

# 3. question_table 데이터 미리 로드 (전체 page_id 공통)
with engine.connect() as conn:
    questions_query = text("""
        SELECT 
            query_id,
            query_text, 
            query_cat, 
            query_keyword 
        FROM question_table
    """)
    raw_questions = conn.execute(questions_query).mappings().fetchall()

    user_queries = []
    for q in raw_questions:
        keywords = q['query_keyword']
        if isinstance(keywords, str):
            try:
                keywords = json.loads(keywords)
            except Exception:
                keywords = [x.strip() for x in keywords.split(',') if x.strip()]
        elif keywords is None:
            keywords = []

        user_queries.append({
            "query_text": q['query_text'],
            "category": q['query_cat'],
            "must_have": keywords
        })

print(f"✅ DB 연결 및 user_queries({len(user_queries)}개) 로드 완료!")

✅ GEOScorer 모델 로드 완료!
✅ DB 연결 및 user_queries(1220개) 로드 완료!


In [9]:
# 1. DB 데이터 추출 함수
MODEL_VERSION = '3.2.0'

def fetch_data_from_db(engine, page_id: int):
    with engine.connect() as conn:
        product_query = text("""
            SELECT 
                product_name, 
                product_cat, 
                text_contents, 
                image_text, 
                json_ld_contents 
            FROM raw_data_table 
            WHERE page_id = :page_id
        """)
        raw_product = conn.execute(product_query, {"page_id": page_id}).mappings().fetchone()

        if not raw_product:
            return None, None, None

        raw_product = dict(raw_product)

        json_ld_data = raw_product['json_ld_contents']
        if isinstance(json_ld_data, str) and json_ld_data.strip():
            try:
                json_ld_data = json.loads(json_ld_data)
            except Exception:
                pass

        image_query = text("""
            SELECT 
                alt_contents 
            FROM image_data_table 
            WHERE page_id = :page_id
            ORDER BY image_sequence ASC
        """)
        raw_images = conn.execute(image_query, {"page_id": page_id}).mappings().fetchall()

        image_list = [
            {
                "alt": img['alt_contents'] or "",
                "is_text_image": False
            }
            for img in raw_images
        ]

        return raw_product, json_ld_data, image_list


# 2. 결과 저장용 리스트
results = []
missing_page_ids = []
error_page_ids = []

# 3. 테스트 범위 순회
for page_id in tqdm(range(4000, 4050), desc="GEO 평가 중"):
    try:
        raw_product, json_ld, image_list = fetch_data_from_db(engine, page_id)
        
        if raw_product is None:
            missing_page_ids.append(page_id)
            continue
        
        # GEO 스코어러 평가 실행
        result = scorer.evaluate_page(
            body_text=raw_product['text_contents'] or "",
            image_text=raw_product['image_text'] or "",
            user_queries=user_queries,
            product_cat=raw_product['product_cat'],
            product_name=raw_product['product_name'],
            json_ld_str=json_ld,
            image_list=image_list
        )
        
        # DB 테이블 스키마 규격 매칭 (메모리 테스트용)
        results.append({
            "page_id": page_id,
            "model_ver": MODEL_VERSION,
            "is_corrected": 0,
            
            # 5대 최종 점수 (FLOAT)
            "text_ratio_score": float(result.text_ratio_score),
            "hybrid_search_score": float(result.hybrid_search.final_score),
            "keyword_stuffing_score": float(result.keyword_stuffing.final_score),
            "json_ld_score": float(result.json_ld.final_score),
            "image_alt_score": float(result.image_alt.avg_score),
            
            # raw_details JSON (세부 내역 완전 포함)
            "raw_details": json.dumps({
                "raw_scores": {
                    "text_ratio_raw_score": float(result.text_ratio_score),
                    "hybrid_search_raw_score": float(result.hybrid_search.avg_combined_score),
                    "keyword_stuffing_raw_score": float(result.keyword_stuffing.raw_score),
                    "json_ld_raw_score": float(result.json_ld.raw_score),
                    "image_alt_raw_score": float(result.image_alt.raw_avg_score)
                },
                "details": {
                    "hybrid_search": {
                        "total_queries_count": result.hybrid_search.total_queries_count,
                        "cat_filtered_count": result.hybrid_search.cat_filtered_count,
                        "must_have_filtered_count": result.hybrid_search.must_have_filtered_count,
                        "avg_lexical_overlap": result.hybrid_search.avg_lexical_overlap,
                        "avg_cosine_sim_raw": result.hybrid_search.avg_cosine_sim_raw,
                    },
                    "keyword_stuffing": {
                        "is_stuffing": result.keyword_stuffing.is_stuffing,
                        "title_raw_score": result.keyword_stuffing.title_raw_score,
                        "body_raw_score": result.keyword_stuffing.body_raw_score,
                        "noun_penalty": result.keyword_stuffing.noun_penalty,
                        "grammatical_penalty": result.keyword_stuffing.grammatical_penalty,
                        "pattern_penalty": result.keyword_stuffing.pattern_penalty,
                        "noun_ratio": result.keyword_stuffing.noun_ratio,
                        "grammatical_ratio": result.keyword_stuffing.grammatical_ratio
                    },
                    "json_ld": {
                        "is_valid": result.json_ld.is_valid,
                        "parsing_score": result.json_ld.parsing_score,
                        "density_score": result.json_ld.density_score,
                        "clothing_score": result.json_ld.clothing_score,
                        "trust_score": result.json_ld.trust_score,
                        "present_attrs_count": result.json_ld.present_attrs_count,
                        "trust_count": result.json_ld.trust_count
                    },
                    "image_alt": {
                        "is_valid": result.image_alt.is_valid,
                        "total_image_count": result.image_alt.total_image_count,
                        "stuffing_image_count": result.image_alt.stuffing_image_count
                    }
                }
            }, ensure_ascii=False),
            
            # 최종 GEO 종합 점수 (INT)
            "geo_score": int(round(result.total_score))
        })

    except Exception as e:
        error_page_ids.append({
            "page_id": page_id,
            "error_msg": str(e)
        })

# 4. 데이터프레임 변환
df_results = pd.DataFrame(results)
df_errors = pd.DataFrame(error_page_ids)

# 5. 결과 및 에러 출력
print(f"🎉 완료! (성공: {len(df_results)}개 / DB 미존재: {len(missing_page_ids)}개 / 에러 발생: {len(df_errors)}개)")

if missing_page_ids:
    print(f"\n📌 DB 미존재 page_id 목록:\n{missing_page_ids}")

if not df_errors.empty:
    print(f"\n⚠️ 처리 중 에러가 발생한 page_id 목록:")
    print(df_errors)

GEO 평가 중:   0%|          | 0/50 [00:00<?, ?it/s]

🎉 완료! (성공: 50개 / DB 미존재: 0개 / 에러 발생: 0개)


In [10]:
pd.DataFrame(results)['raw_details'][0]

'{"raw_scores": {"text_ratio_raw_score": 1.0, "hybrid_search_raw_score": 0.4028, "keyword_stuffing_raw_score": 0.609, "json_ld_raw_score": 0.175, "image_alt_raw_score": 0.244}, "details": {"hybrid_search": {"total_queries_count": 1220, "cat_filtered_count": 401, "must_have_filtered_count": 111, "avg_lexical_overlap": 0.1899, "avg_cosine_sim_raw": 0.6158}, "keyword_stuffing": {"is_stuffing": false, "title_raw_score": 0.55, "body_raw_score": 0.746, "noun_penalty": 0.0, "grammatical_penalty": 0.089, "pattern_penalty": 0.165, "noun_ratio": 0.32, "grammatical_ratio": 0.15}, "json_ld": {"is_valid": true, "parsing_score": 0.7, "density_score": 0.0, "clothing_score": 0.0, "trust_score": 0.0, "present_attrs_count": 0, "trust_count": 0}, "image_alt": {"is_valid": true, "total_image_count": 61, "stuffing_image_count": 47}}}'